## OGX Vision API
This notebook demonstrates how to use the OGX inference adapter to interact with
SambaNova cloud-based vision models using the OpenAI-compatible chat completions API.

Please refer to the [OGX quickstart documentation](https://ogx-ai.github.io/docs/getting_started/quickstart) for further details.

## Setup

In [1]:
# Imports
import base64
import mimetypes
from ogx_client import OgxClient
from termcolor import cprint


In [ ]:
# Select model
model = 'sambanova/gemma-4-31B-it'
# Select image path
image_path='../../images/SambaNova-dark-logo-1.png'


In [3]:
# Initialize client
OGX_PORT = 8321
client = OgxClient(base_url=f"http://localhost:{OGX_PORT}")


## Helper functions
Some utility functions to handle image processing and API interaction.

In [4]:
def encode_image_to_data_url(file_path: str) -> str:
    """
    Encode an image file to a data URL.

    Args:
        file_path: Path to the image file

    Returns:
        Data URL string
    """
    mime_type, _ = mimetypes.guess_type(file_path)
    if mime_type is None:
        raise ValueError("Could not determine MIME type of the file")

    with open(file_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")

    return f"data:{mime_type};base64,{encoded_string}"


## Chat with the image
The code below uses the Llama Stack Chat API to interact with the LLM.


In [5]:
def process_image_chat(client: OgxClient, image_path: str, stream: bool = False):
    """
    Process an image through the OGX Chat Completions API.

    Args:
        client: Initialized OgxClient.
        image_path: Path to image file.
        stream: Whether to stream the response.
    """
    data_url = encode_image_to_data_url(image_path)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text", "text": "What does this image represent?"},
            ],
        },
    ]

    cprint("User> Sending image for analysis...", "green")

    if stream:
        cprint("> Response: ", "cyan", end="")
        for chunk in client.chat.completions.create(
            model=model,
            messages=messages,
            stream=True,
        ):
            content = chunk.choices[0].delta.content or ""
            cprint(content, "cyan", end="", flush=True)
        print()
    else:
        response = client.chat.completions.create(model=model, messages=messages)
        cprint(f"> Response: {response.choices[0].message.content}", "cyan")


In [6]:
# Chat with the iamge
process_image_chat(client=client, image_path=image_path, stream=True)